In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
df = pd.read_csv("Preprocessed_HousePriceData.csv")

In [3]:
df["Garage"] = df["Garage"].map({1: "Yes", 0: "No"})

In [4]:
X = df.drop(columns=['Price', 'Id'])
y = df['Price']

In [5]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [6]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [7]:
feature_selector = SelectKBest(score_func=f_regression)

In [8]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', feature_selector),
    ('model', LinearRegression()) 
])

In [9]:
param_grid = [
    {
        'selector__k': [10, 20, 'all'],
        'model': [LinearRegression()]
    },
    {
        'selector__k': [10, 20, 50],
        'model': [Ridge()],
        'model__alpha': [0.1, 1.0, 10.0]
    },
    {
        'selector__k': [10, 20, 50],
        'model': [Lasso(max_iter=5000)],
        'model__alpha': [0.001, 0.01, 0.1, 1.0]
    },
    {
        'selector__k': [20, 50, 'all'],
        'model': [RandomForestRegressor(random_state=42)],
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20]
    },
    {
        'selector__k': [20, 50, 'all'],
        'model': [GradientBoostingRegressor(random_state=42)],
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [3, 5]
    }
]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [11]:
grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 66 candidates, totalling 330 fits


,estimator,Pipeline(step...egression())])
,param_grid,"[{'model': [LinearRegression()], 'selector__k': [10, 20, ...]}, {'model': [Ridge()], 'model__alpha': [0.1, 1.0, ...], 'selector__k': [10, 20, ...]}, ...]"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [12]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (R2):", grid_search.best_score_)
print("Test Score (R2):", grid_search.score(X_test, y_test))

Best Parameters: {'model': Ridge(), 'model__alpha': 10.0, 'selector__k': 10}
Best CV Score (R2): -0.008611047734519817
Test Score (R2): -0.006451646562506408


In [13]:
best_model = grid_search.best_estimator_

In [14]:
with open("Best_HousePrice_Model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Model saved successfully as Best_HousePrice_Model.pkl")

Model saved successfully as Best_HousePrice_Model.pkl
